# IE-OPT-03 · 设施选址与产能分配（PySCIPOpt 原创教学案例）

**场景**：三个候选区域维修中心服务四个需求区。决定启用哪些维修中心以及每个区域由谁服务，在产能限制下最小化固定运营与运输费用。

本案例以 [PySCIPOpt 官方设施选址示例 flp.py](https://github.com/scipopt/PySCIPOpt/blob/master/examples/finished/flp.py) 为**学习参考**，但模型、数据、测试与表述为独立课程实现，未复制官方代码。

特别注意：这里采用 **single-source** 指派（每个客户整体交由一家设施），官方 flp.py 使用可分割连续运输量模型，两者的可行域不同。该案例为合成教学数据，不是实际企业成本。

## 1. 数学模型

- 二元变量 y_j：设施 j 是否开放；二元变量 x_ij：需求区 i 是否整体交由 j 服务。
- 最小化固定开设成本 + 各客户运量×单位运输成本。
- 每个需求区恰好由一个设施服务；每个设施承担的需求不得超过容量；未开放设施不得接受指派。

关键约束：Σ_j x_ij = 1；Σ_i d_i x_ij ≤ C_j y_j；x_ij ≤ y_j。

本案例不处理排队、动态到达、运输延误和故障。最优静态分配不自动保证动态运行表现最好。

In [ ]:
from itertools import product
from pyscipopt import Model, quicksum

sites = ["成都", "重庆", "昆明"]
customers = ["区域A", "区域B", "区域C", "区域D"]
demand = [25, 35, 20, 30]  # 合成需求：件/周期
capacity = [90, 80, 85]    # 每周期可处理上限
opening = [95, 80, 90]     # 固定成本
cost = [
    [2, 5, 8],
    [5, 2, 7],
    [6, 4, 3],
    [8, 5, 2]
]  # 单件运输成本：行=需求区，列=候选设施
assert sum(demand) <= sum(capacity)

## 2. 用 PySCIPOpt 建立和求解混合整数线性规划

可尝试修改固定成本或容量，观察设施启用方案如何改变；求解状态不是 optimal 时不能直接读取最优值。

In [ ]:
def solve_facility(demand, capacity, opening, cost):
    n, k = len(demand), len(capacity)
    model = Model("IE-OPT-03 single-source facility")
    model.hideOutput(True)
    y = {j: model.addVar(vtype="B", name=f"open_{j}") for j in range(k)}
    x = {(i, j): model.addVar(vtype="B", name=f"serve_{i}_{j}")
         for i in range(n) for j in range(k)}
    for i in range(n):
        model.addCons(quicksum(x[i, j] for j in range(k)) == 1)
    for j in range(k):
        model.addCons(quicksum(demand[i] * x[i, j] for i in range(n))
                      <= capacity[j] * y[j])
    for i in range(n):
        for j in range(k):
            model.addCons(x[i, j] <= y[j])
    model.setObjective(
        quicksum(opening[j] * y[j] for j in range(k)) +
        quicksum(demand[i] * cost[i][j] * x[i, j]
                 for i in range(n) for j in range(k)), "minimize")
    model.optimize()
    status = str(model.getStatus())
    if status != "optimal":
        raise RuntimeError(f"Solver status is {status}; optimality not certified")
    allocation = [next(j for j in range(k)
                       if model.getVal(x[i, j]) > .5) for i in range(n)]
    opened = [j for j in range(k) if model.getVal(y[j]) > .5]
    return {"objective": float(model.getObjVal()),
            "allocation": allocation, "opened": opened}

result = solve_facility(demand, capacity, opening, cost)
print("最小费用:", result["objective"])
print("启用设施:", [sites[j] for j in result["opened"]])
print("客户分配:", {customers[i]: sites[result["allocation"][i]]
                    for i in range(len(customers))})

## 3. 独立穷举 Oracle 验证（81 种指派，不依赖 SCIP 的求解结果）

枚举 3^4=81 种分配，逐一检查容量；启用设施为有至少一名客户指派的设施。由于固定成本非负，在此模型中最优解不需要开启空闲设施。这给出一个独立且可检查的最优目标对照。

In [ ]:
def brute_force_oracle(demand, capacity, opening, cost):
    best = float("inf")
    best_plan = None
    for allocation in product(range(len(capacity)), repeat=len(demand)):
        load = [sum(demand[i] for i in range(len(demand))
                    if allocation[i] == j) for j in range(len(capacity))]
        if any(load[j] > capacity[j] for j in range(len(capacity))):
            continue
        used = [j for j in range(len(capacity)) if load[j] > 0]
        score = (sum(opening[j] for j in used) +
                 sum(demand[i] * cost[i][allocation[i]]
                     for i in range(len(demand))))
        if score < best:
            best, best_plan = score, allocation
    return best, best_plan

oracle_obj, oracle_plan = brute_force_oracle(demand, capacity, opening, cost)
print("独立穷举目标:", oracle_obj)
print("一种最优分配:", oracle_plan)
assert abs(result["objective"] - oracle_obj) < 1e-6
for j in range(len(sites)):
    actual = sum(demand[i] for i, assigned in enumerate(result["allocation"])
                 if assigned == j)
    assert actual <= capacity[j]
    assert (j in result["opened"]) or actual == 0
print("通过：求解器目标与独立穷举一致；容量及开启约束满足")

## 4. 从“优化”走向“仿真”

1. 固定这份静态分配，若客户需求每天随机到达且维修服务时间不确定，中心会出现怎样的等待？
2. 把需求改为多个高/低负荷场景，分别求解静态最优，再测试方案在未见场景的表现；切勿把测试数据作为决策时已知信息。
3. 利用课程 [IE-DES-01 排队系统](../case-studio.html) 理解等待和工位数，或用 [IE-MC-02 库存仿真](../inventory-lab.html) 对比优化与模拟的假设差别。
4. 课堂拓展：允许客户需求可分割时，须将 x_ij 改成连续流量并重写守恒约束，求解结果不一定与当前整数指派模型相同。

**课程性质**：此 Notebook 是可运行示范与独立验证教程，未接入正式 L2 私有评分；正式课程成绩以教师控制台的已授权评分任务为准。